# <font color="#418FDE" size="6.5" uppercase>**Evaluation Design**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Design evaluation schemes that avoid leakage and respect data structure. 
- Use nested cross-validation and permutation tests for more reliable conclusions. 
- Interpret multi-metric scores, timing results, and fold-to-fold variation. 


## **1. Leakage Control**

### **1.1. Randomized Shuffling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_B/image_01_01.jpg?v=1784033741" width="250">



>* Shuffle arbitrary order to reduce split bias
>* Preserve meaningful structure to prevent leakage

>* Hidden dependencies make random splits leak information
>* Split by independent units, not rows

>* Use controlled random splits for reproducibility
>* Match shuffling to real prediction structure



In [ ]:
#@title Python Code - Randomized Shuffling

# This example compares shuffled and unshuffled splits.
# Row order can accidentally bias evaluation results.
# Shuffling helps only when rows are independent.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Create a small classification dataset with deterministic randomness.
features, target = make_classification(
    n_samples=300,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    random_state=42,
)

# Sort rows by target to mimic an accidentally ordered export.
row_order = np.argsort(target)
features_ordered = features[row_order]
target_ordered = target[row_order]

# Validate that the ordered dataset has the expected shape.
if features_ordered.shape != (300, 6):
    raise ValueError("Expected 300 rows and 6 feature columns.")

# Split once without shuffling and once with controlled shuffling.
plain_split = train_test_split(
    features_ordered,
    target_ordered,
    test_size=0.3,
    shuffle=False,
)

shuffled_split = train_test_split(
    features_ordered,
    target_ordered,
    test_size=0.3,
    shuffle=True,
    stratify=target_ordered,
    random_state=42,
)

# Train the same simple model on each split.
plain_model = LogisticRegression(max_iter=500, random_state=42)
plain_model.fit(plain_split[0], plain_split[2])

shuffled_model = LogisticRegression(max_iter=500, random_state=42)
shuffled_model.fit(shuffled_split[0], shuffled_split[2])

# Compare class balance and accuracy for both evaluation designs.
plain_test_rate = np.mean(plain_split[3])
shuffled_test_rate = np.mean(shuffled_split[3])
plain_accuracy = accuracy_score(plain_split[3], plain_model.predict(plain_split[1]))
shuffled_accuracy = accuracy_score(
    shuffled_split[3],
    shuffled_model.predict(shuffled_split[1]),
)

print("scikit-learn version:", sklearn.__version__)
print("Unshuffled test positive rate:", round(plain_test_rate, 2))
print("Shuffled test positive rate:", round(shuffled_test_rate, 2))
print("Unshuffled accuracy:", round(plain_accuracy, 2))
print("Shuffled accuracy:", round(shuffled_accuracy, 2))
print("Lesson: shuffle arbitrary row order, but protect real groups or time.")



### **1.2. Stratification Boundaries**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_B/image_01_02.jpg?v=1784033737" width="250">



>* Preserve class proportions across evaluation folds
>* Avoid stratifying with unavailable future information

>* Avoid stratifying on too many variables
>* Choose boundaries that preserve realistic evaluation

>* Keep related records in the same split
>* Stratify only after preventing leakage



### **1.3. Group Leakage**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_B/image_01_03.jpg?v=1784033739" width="250">



>* Related records across folds cause leakage
>* Scores become overly optimistic, not generalizable

>* Choose groups matching the real prediction goal
>* Keep each group within one split

>* Group splits give more realistic performance estimates
>* Remove group-identifying features that create shortcuts



In [ ]:
#@title Python Code - Group Leakage

# This example shows group leakage in validation.
# Related rows must stay in one fold.
# Group-aware scores are usually more realistic.

import numpy as np
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

# Create repeated observations from the same hidden groups.
rng = np.random.default_rng(42)
n_groups = 60
rows_per_group = 4

# Each group has a stable signature and one shared label.
group_ids = np.repeat(np.arange(n_groups), rows_per_group)
group_signal = rng.normal(size=n_groups)
group_label = (group_signal > 0).astype(int)

# Rows from the same group look very similar.
noise = rng.normal(scale=0.15, size=n_groups * rows_per_group)
feature_one = group_signal[group_ids] + noise
feature_two = rng.normal(size=n_groups * rows_per_group)

# Build the feature matrix and target vector.
X = np.column_stack([feature_one, feature_two])
y = group_label[group_ids]

# Validate the small teaching dataset before modeling.
if X.shape != (n_groups * rows_per_group, 2):
    raise ValueError("Unexpected feature shape for this example.")

# Use one simple classifier for both evaluation designs.
model = LogisticRegression(random_state=42, max_iter=200)
random_cv = KFold(n_splits=5, shuffle=True, random_state=42)
group_cv = GroupKFold(n_splits=5)

# Random folds can place related rows on both sides.
random_scores = cross_val_score(model, X, y, cv=random_cv)
group_scores = cross_val_score(model, X, y, cv=group_cv, groups=group_ids)

# Count leaked groups in the first random split.
train_index, test_index = next(random_cv.split(X, y))
train_groups = set(group_ids[train_index])
test_groups = set(group_ids[test_index])
leaked_groups = len(train_groups.intersection(test_groups))

print("scikit-learn version:", sklearn.__version__)
print("Leaked groups in one random fold:", leaked_groups)
print("Random CV mean accuracy:", round(random_scores.mean(), 3))
print("GroupKFold mean accuracy:", round(group_scores.mean(), 3))
print("Lesson: keep each group entirely inside one fold.")



## **2. Robust Model Evaluation**

### **2.1. Nested Cross Validation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_B/image_02_01.jpg?v=1784033728" width="250">



>* Separate model selection from final assessment
>* Inner tunes; outer estimates unbiased performance

>* Outer folds mimic new deployment tests
>* Inner tuning never touches outer test data

>* Higher cost, more trustworthy performance estimates
>* Outer-fold variation shows split sensitivity



### **2.2. Permutation Significance Testing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_B/image_02_02.jpg?v=1784033726" width="250">



>* Compare real scores with shuffled-label scores
>* Strong real performance suggests genuine signal

>* Shuffled labels create a chance-performance baseline
>* Compare real scores against shuffled-score distributions

>* Test the full evaluation workflow
>* Interpret significance as chance-based evidence



In [ ]:
#@title Python Code - Permutation Significance Testing

# Demonstrate permutation testing for model evaluation.
# Compare real labels with shuffled labels.
# Expect real accuracy above chance scores.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic dataset shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Use one pipeline so scaling happens inside each fold.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Reuse the same stratified folds for fair comparisons.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
real_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
real_mean = real_scores.mean()

# Shuffle labels repeatedly to create chance-level scores.
rng = np.random.default_rng(42)
permutation_scores = []
for repeat in range(30):
    shuffled_y = rng.permutation(y)
    scores = cross_val_score(model, X, shuffled_y, cv=cv, scoring="accuracy")
    permutation_scores.append(scores.mean())

# Estimate the permutation p-value with a small-sample correction.
permutation_scores = np.array(permutation_scores)
count_at_least_real = np.sum(permutation_scores >= real_mean)
p_value = (count_at_least_real + 1) / (len(permutation_scores) + 1)

print("scikit-learn version:", sklearn.__version__)
print("Real-label mean accuracy:", round(real_mean, 3))
print("Permutation p-value:", round(p_value, 3))

# Plot the chance distribution and the real score.
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(permutation_scores, bins=10, color="lightgray", edgecolor="black")
ax.axvline(real_mean, color="red", linewidth=2, label="real labels")
ax.set_title("Permutation test: real score versus shuffled labels")
ax.set_xlabel("Cross-validated accuracy")
ax.set_ylabel("Number of permutations")
ax.legend()
plt.show()



### **2.3. Multi Metric Evaluation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_B/image_02_03.jpg?v=1784033730" width="250">



>* Use multiple metrics to reveal hidden weaknesses
>* Compare trade-offs beyond one average score

>* Tune using one primary inner-loop metric
>* Report broader outer-loop strengths and trade-offs

>* Test whether scores beat chance performance
>* Compare metrics, uncertainty, and fold consistency



In [ ]:
#@title Python Code - Multi Metric Evaluation

# Compare several metrics on identical validation folds.
# Multi metric evaluation reveals practical tradeoffs.
# Results show averages and fold variation.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Build preprocessing inside the pipeline to avoid leakage.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Use the same folds for every metric.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Score complementary questions about the same predictions.
scoring = {
    "recall": make_scorer(recall_score),
    "precision": make_scorer(precision_score),
    "roc_auc": "roc_auc",
}

# Cross validate once and collect all requested metrics.
results = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

# Summarize average performance and fold-to-fold variation.
summary = pd.DataFrame({
    "metric": ["recall", "precision", "roc_auc"],
    "mean": [
        np.mean(results["test_recall"]),
        np.mean(results["test_precision"]),
        np.mean(results["test_roc_auc"]),
    ],
    "std": [
        np.std(results["test_recall"]),
        np.std(results["test_precision"]),
        np.std(results["test_roc_auc"]),
    ],
})

# Display concise results for beginner interpretation.
print("scikit-learn version:", sklearn.__version__)
print("Same 5 folds, three metrics:")
print(summary.round(3).to_string(index=False))
print("Mean fit time per fold:", round(np.mean(results["fit_time"]), 3), "seconds")



## **3. Interpreting CV Results**

### **3.1. Fit Score Timing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_B/image_03_01.jpg?v=1784033732" width="250">



>* Compare model quality with training cost
>* Consider prediction speed for real-world use

>* Complex models increase repeated training costs
>* Slow folds reveal hidden operational issues

>* Score time measures prediction cost after training
>* Balance accuracy gains against timing constraints



In [ ]:
#@title Python Code - Fit Score Timing

# Compare cross-validation scores with timing costs.
# Fit time measures repeated training effort.
# Score time measures validation prediction effort.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Build one simple pipeline to avoid preprocessing leakage.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Cross-validation returns scores and timing for each fold.
cv_results = cross_validate(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy",
    return_train_score=False
)

# Summarize fold-to-fold variation in a compact table.
summary = pd.DataFrame(
    {
        "accuracy": cv_results["test_score"],
        "fit_seconds": cv_results["fit_time"],
        "score_seconds": cv_results["score_time"],
    }
)

# Convert timing to milliseconds for easier reading.
summary["fit_ms"] = summary["fit_seconds"] * 1000
summary["score_ms"] = summary["score_seconds"] * 1000
shown = summary[["accuracy", "fit_ms", "score_ms"]].round(3)

print("scikit-learn version:", sklearn.__version__)
print(shown.to_string(index=False))
print("Mean accuracy:", round(summary["accuracy"].mean(), 3))
print("Mean fit time ms:", round(summary["fit_ms"].mean(), 2))
print("Mean score time ms:", round(summary["score_ms"].mean(), 2))



### **3.2. Pipeline Wide Validation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_B/image_03_02.jpg?v=1784033733" width="250">



>* Validate the entire workflow, not just model
>* Refit preprocessing inside each training fold

>* Link metrics to the full workflow
>* Validate steps as future cases experience them

>* Fit time shows workflow practicality
>* Fold variation reveals pipeline stability



In [ ]:
#@title Python Code - Pipeline Wide Validation

# This script compares leaky and pipeline-wide validation.
# It shows fold scores and timing together.
# The result highlights workflow-level interpretation.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_classification

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Create many noisy features to make leakage visible.
features, target = make_classification(
    n_samples=600,
    n_features=80,
    n_informative=6,
    n_redundant=4,
    random_state=42,
)

# Validate the basic shape before modeling.
if features.shape != (600, 80):
    raise ValueError("Expected 600 rows and 80 columns.")

# This selector is incorrectly fit before cross-validation.
leaky_selector = SelectKBest(score_func=f_classif, k=10)
leaky_features = leaky_selector.fit_transform(features, target)

# The model is simple, deterministic, and beginner-friendly.
model = LogisticRegression(max_iter=1000, random_state=42)
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Cross-validation now sees features chosen using all rows.
leaky_results = cross_validate(
    model,
    leaky_features,
    target,
    cv=folds,
    scoring="accuracy",
    return_train_score=False,
)

# The pipeline refits scaling and selection inside each fold.
proper_pipeline = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("select", SelectKBest(score_func=f_classif, k=10)),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

# This score estimates the whole deployable workflow.
proper_results = cross_validate(
    proper_pipeline,
    features,
    target,
    cv=folds,
    scoring="accuracy",
    return_train_score=False,
)

# Summarize average score, variation, and fitting time.
summary = pd.DataFrame(
    {
        "mean_acc": [leaky_results["test_score"].mean(), proper_results["test_score"].mean()],
        "std_acc": [leaky_results["test_score"].std(), proper_results["test_score"].std()],
        "mean_fit_s": [leaky_results["fit_time"].mean(), proper_results["fit_time"].mean()],
    },
    index=["Leaky preselection", "Pipeline-wide CV"],
)

# Print concise results for interpretation.
print("scikit-learn version:", sklearn.__version__)
print(summary.round(3).to_string())
print("Pipeline-wide CV evaluates preprocessing plus the model inside each fold.")



### **3.3. Fold Score Variation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_B/image_03_03.jpg?v=1784033735" width="250">



>* Fold scores reveal consistency beyond averages
>* Large drops may expose hidden weaknesses

>* Compare variation size with data structure
>* Large differences reveal reliability concerns

>* Match variation tolerance to model risk
>* Prefer dependable folds over occasional failures



In [ ]:
#@title Python Code - Fold Score Variation

# This example compares cross-validation fold scores.
# Fold variation reveals stability beyond average accuracy.
# The plot highlights dependable and unstable performance.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Create a small classification dataset with deterministic randomness.
features, target = make_classification(
    n_samples=600,
    n_features=10,
    n_informative=4,
    n_redundant=2,
    weights=[0.7, 0.3],
    class_sep=0.8,
    random_state=42,
)

# Validate the basic shape before modeling.
if features.shape[0] != target.shape[0]:
    raise ValueError("Features and target must have matching rows.")

# Build a leakage-safe pipeline for each training fold.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=500, random_state=42),
)

# Stratified folds keep class proportions similar across folds.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Collect fold-level accuracy and F1 scores.
scores = cross_validate(
    model,
    features,
    target,
    cv=cv,
    scoring={"accuracy": "accuracy", "f1": "f1"},
)

accuracy_scores = scores["test_accuracy"]
f1_scores = scores["test_f1"]
fold_numbers = np.arange(1, len(accuracy_scores) + 1)

# Summaries show average performance and fold-to-fold spread.
print("scikit-learn version:", sklearn.__version__)
print("Accuracy mean and range:", round(accuracy_scores.mean(), 3), round(np.ptp(accuracy_scores), 3))
print("F1 mean and range:", round(f1_scores.mean(), 3), round(np.ptp(f1_scores), 3))

# Plot each fold to make variation visible.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(fold_numbers, accuracy_scores, marker="o", label="Accuracy")
ax.plot(fold_numbers, f1_scores, marker="s", label="F1 score")
ax.axhline(accuracy_scores.mean(), linestyle="--", color="C0", alpha=0.5)
ax.axhline(f1_scores.mean(), linestyle="--", color="C1", alpha=0.5)

# Labels connect the visual spread to cross-validation folds.
ax.set_title("Fold Score Variation Across Five CV Folds")
ax.set_xlabel("Cross-validation fold")
ax.set_ylabel("Score")
ax.set_xticks(fold_numbers)
ax.set_ylim(0, 1)
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Evaluation Design**</font>


In this lecture, you learned to:
- Design evaluation schemes that avoid leakage and respect data structure. 
- Use nested cross-validation and permutation tests for more reliable conclusions. 
- Interpret multi-metric scores, timing results, and fold-to-fold variation. 

In the next Module (Module 28), we will go over 'Tuning Curves'